### FNN trained on data encoded with 0.5 scaling factor

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics import roc_curve
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, roc_auc_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
import json
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.layers import Input

In [2]:
# Set random seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

In [3]:
# Function to load data
def load_data(matrix_folder, label_file):
    # Load labels
    with open(label_file, 'r') as f:
        labels = np.array([int(line.strip()) for line in f])

    # Load matrices
    matrices = []
    for file in sorted(os.listdir(matrix_folder)):
        if file.endswith('.csv'):
            matrix_path = os.path.join(matrix_folder, file)
            matrix = pd.read_csv(matrix_path, header=None, skiprows=1).values
            matrices.append(matrix)

    matrices = np.array(matrices)
    return matrices, labels

In [ ]:
# Function for 10-fold cross-validation training
def cross_validate_model(X, y, n_splits=10):
    X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    scaler = StandardScaler()
    X_trainval = scaler.fit_transform(X_trainval)
    X_test = scaler.transform(X_test)

    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    fold_accuracies = []
    fold_roc_aucs = []
    fold_reports = []
    fold_confusion_matrices = []

    fold_num = 1
    for train_index, val_index in kfold.split(X_trainval):
        print(f"\nTraining Fold {fold_num}/{n_splits}...")
        X_train, X_val = X_trainval[train_index], X_trainval[val_index]
        y_train, y_val = y_trainval[train_index], y_trainval[val_index]

        model = Sequential([
            Input(shape=(X_train.shape[1],)),
            Dense(64, activation='sigmoid', kernel_regularizer=l2(0.001)),
            Dense(32, activation='sigmoid', kernel_regularizer=l2(0.001)),
            Dropout(0.4),
            Dense(16, activation='sigmoid', kernel_regularizer=l2(0.001)),
            Dropout(0.4),
            Dense(1, activation='sigmoid')  # Binary classification
        ]) 

        model.compile(optimizer=Adam(learning_rate=0.001),
                      loss='binary_crossentropy',
                      metrics=['accuracy'])


        model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=15,
            batch_size=32,
            verbose=1
        )

        # Predict probabilities and labels for the validation set
        y_test_probs = model.predict(X_test, verbose=0)
        y_test_pred = (y_test_probs > 0.5).astype(int)


        # Calculate metrics for the fold
        accuracy = accuracy_score(y_test, y_test_pred)
        roc_auc = roc_auc_score(y_test, y_test_probs)
        classification_rep = classification_report(y_test, y_test_pred, target_names=['Negative', 'Positive'], output_dict=True)
        confusion_mat = confusion_matrix(y_test, y_test_pred)

        # Append results for the fold
        fold_accuracies.append(accuracy)
        fold_roc_aucs.append(roc_auc)
        fold_reports.append(classification_rep)
        fold_confusion_matrices.append(confusion_mat)

        # Calculate average metrics across all folds
        avg_accuracy = np.mean(fold_accuracies)
        avg_auc = np.mean(fold_roc_aucs)

        print(f"Fold {fold_num} Accuracy: {accuracy:.4f}, ROC-AUC: {roc_auc:.4f}")
        fold_num += 1
        
    avg_classification_report = {
        'Positive': {
            'accuracy': float(avg_accuracy),
            'auc': float(avg_auc),
            'precision': float(np.mean([r['Positive']['precision'] for r in fold_reports])),
            'recall': float(np.mean([r['Positive']['recall'] for r in fold_reports])),
            'f1-score': float(np.mean([r['Positive']['f1-score'] for r in fold_reports]))
        },
        'Negative': {
            'accuracy': float(avg_accuracy),
            'auc': float(avg_auc),
            'precision': float(np.mean([r['Negative']['precision'] for r in fold_reports])),
            'recall': float(np.mean([r['Negative']['recall'] for r in fold_reports])),
            'f1-score': float(np.mean([r['Negative']['f1-score'] for r in fold_reports]))
        }
    }

    print("\nAverage Classification Report (across all folds):")
    print(avg_classification_report)


    return fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report


Saving the results into a JSON

In [5]:
def save_classification_report(dataset_name, avg_classification_report):
    results_file = "reports\classification_reports2_ffnn.json"

    # Add dataset name to the report
    report_to_save = {
        "dataset": dataset_name,
        "report": avg_classification_report
    }

    # Load existing results if the file exists
    if os.path.exists(results_file):
        with open(results_file, "r") as f:
            existing_results = json.load(f)
    else:
        existing_results = []

    # Append new results
    existing_results.append(report_to_save)

    # Save updated results
    with open(results_file, "w") as f:
        json.dump(existing_results, f, indent=4)

    print(f"Classification report saved to {results_file}")

In [6]:
def save_results(dataset_name, fold_accuracies, fold_roc_aucs, fold_reports):
    results_file = "reports/classification_results2_FNN.json"

    # Extract precision, recall, and f1-score for each fold
    fold_metrics = []
    for report in fold_reports:
        fold_metrics.append({
            "Negative": {
                "precision": report["Negative"]["precision"],
                "recall": report["Negative"]["recall"],
                "f1-score": report["Negative"]["f1-score"]
            },
            "Positive": {
                "precision": report["Positive"]["precision"],
                "recall": report["Positive"]["recall"],
                "f1-score": report["Positive"]["f1-score"]
            }
        })

    # Convert results to a dictionary
    results_dict = {
        "dataset": dataset_name,
        "accuracies": fold_accuracies,
        "roc_aucs": fold_roc_aucs,
        "fold_metrics": fold_metrics
    }

    # Load existing results if the file exists
    if os.path.exists(results_file):
        with open(results_file, "r") as f:
            existing_results = json.load(f)
    else:
        existing_results = []

    # Append new results
    existing_results.append(results_dict)

    # Save updated results
    with open(results_file, "w") as f:
        json.dump(existing_results, f, indent=4)

    print(f"Results saved to {results_file}")



RES 50

In [7]:
matrix_folder_antiinflam = 'data/matrices/aip_antiinflam_matrix'  
label_file_antiinflam = 'data/labels/aip_antiinflam.txt' 
matrices, labels = load_data(matrix_folder_antiinflam, label_file_antiinflam)

# Flatten matrices if necessary
X = matrices.reshape(matrices.shape[0], -1)
y = labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)


Training Fold 1/10...
Epoch 1/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - accuracy: 0.5768 - loss: 0.8805 - val_accuracy: 0.5941 - val_loss: 0.7500
Epoch 2/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.5555 - loss: 0.7754 - val_accuracy: 0.5941 - val_loss: 0.7343
Epoch 3/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.5543 - loss: 0.7459 - val_accuracy: 0.5941 - val_loss: 0.7230
Epoch 4/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5796 - loss: 0.7414 - val_accuracy: 0.5941 - val_loss: 0.7129
Epoch 5/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5834 - loss: 0.7167 - val_accuracy: 0.5941 - val_loss: 0.7027
Epoch 6/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6320 - loss: 0.6990 - val_accuracy: 0.6353 - val_loss: 0.6898
Epoch 7/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.6788 - loss: 0.6684 - val_accuracy: 0.6471 - val_loss: 0.6813
Epoch 8/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7152 - loss: 0.6347 - val_

In [8]:
save_results("aip_antiinflam_50", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/classification_results2_FNN.json


In [9]:
save_classification_report("aip_antiinflam_50", avg_classification_report)

Classification report saved to reports\classification_reports2_ffnn.json


In [10]:

# Load dataset
matrix_folder_antipb = 'data/matrices/amp_antibp_matrix'
label_file_antipb = 'data/labels/amp_antibp.txt'
matrices, labels = load_data(matrix_folder_antipb, label_file_antipb)# Preprocess data
# Flatten matrices if necessary
X = matrices.reshape(matrices.shape[0], -1)
y = labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - accuracy: 0.5096 - loss: 0.9059 - val_accuracy: 0.4928 - val_loss: 0.7917
Epoch 2/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.4931 - loss: 0.8207 - val_accuracy: 0.8116 - val_loss: 0.7532
Epoch 3/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.5405 - loss: 0.7826 - val_accuracy: 0.8406 - val_loss: 0.7306
Epoch 4/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6094 - loss: 0.7370 - val_accuracy: 0.8406 - val_loss: 0.7067
Epoch 5/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.6541 - loss: 0.7079 - val_accuracy: 0.8406 - val_loss: 0.6753
Epoch 6/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7189 - loss: 0.6693 - val_accuracy: 0.8406 - val_loss: 0.6375
Epoch 7/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7564 - loss: 0.6281 - val_accuracy: 0.8406 - val_loss: 0.5993
Epoch 8/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8276 - loss: 0.5518 - v

In [11]:
save_results("amp_antibp_50", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/classification_results2_FNN.json


In [12]:
save_classification_report("amp_antibp_50", avg_classification_report)

Classification report saved to reports\classification_reports2_ffnn.json


In [13]:
# Load dataset
matrix_folder_antipb2 = 'data/matrices/amp_antibp2_matrix' 
label_file_antipb2 = 'data/labels/amp_antibp2.txt'  
matrices, labels = load_data(matrix_folder_antipb2, label_file_antipb2)
# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels


# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.5027 - loss: 0.8701 - val_accuracy: 0.8250 - val_loss: 0.7553
Epoch 2/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5437 - loss: 0.7751 - val_accuracy: 0.8188 - val_loss: 0.7193
Epoch 3/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6133 - loss: 0.7141 - val_accuracy: 0.8125 - val_loss: 0.6780
Epoch 4/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6926 - loss: 0.6813 - val_accuracy: 0.8250 - val_loss: 0.6314
Epoch 5/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7463 - loss: 0.6238 - val_accuracy: 0.8000 - val_loss: 0.6061
Epoch 6/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7985 - loss: 0.5773 - val_accuracy: 0.8000 - val_loss: 0.5936
Epoch 7/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8473 - loss: 0.5247 - val_accuracy: 0.7875 - val_loss: 0.5893
Epoch 8/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8616 - loss: 0.4929 - val_accu

In [14]:
save_results("amp_antibp2_50", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/classification_results2_FNN.json


In [15]:
save_classification_report("amp_antibp2_50", avg_classification_report)

Classification report saved to reports\classification_reports2_ffnn.json


In [16]:
# Load dataset
matrix_folder_csamp = 'data/matrices/amp_csamp_matrix'  
label_file_csamp = 'data/labels/amp_csamp.txt'  
matrices, labels = load_data(matrix_folder_csamp, label_file_csamp)
# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 4s 121ms/step - accuracy: 0.5392 - loss: 0.9521 - val_accuracy: 0.4286 - val_loss: 0.9476
Epoch 2/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.5297 - loss: 0.9126 - val_accuracy: 0.4286 - val_loss: 0.8887
Epoch 3/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.5260 - loss: 0.8461 - val_accuracy: 0.4286 - val_loss: 0.8446
Epoch 4/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.5575 - loss: 0.7986 - val_accuracy: 0.4286 - val_loss: 0.8143
Epoch 5/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.5096 - loss: 0.8018 - val_accuracy: 0.4286 - val_loss: 0.7945
Epoch 6/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.4630 - loss: 0.8109 - val_accuracy: 0.5238 - val_loss: 0.7815
Epoch 7/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.5012 - loss: 0.7755 - val_accuracy: 0.6190 - val_loss: 0.7725
Epoch 8/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.6019 - loss: 0.7557 - val_accuracy: 0.

In [17]:
save_results("amp_csamp_50", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/classification_results2_FNN.json


In [18]:
save_classification_report("amp_csamp_50", avg_classification_report)

Classification report saved to reports\classification_reports2_ffnn.json


In [19]:
# Load dataset
matrix_folder_hivddi = 'data/matrices/hiv_ddi_matrix' 
label_file_hivddi = 'data/labels/hiv_ddi.txt'  
matrices, labels = load_data(matrix_folder_hivddi, label_file_hivddi)

In [20]:
# Load dataset
matrix_folder_hivddi = 'data/matrices/hiv_ddi_matrix' 
label_file_hivddi = 'data/labels/hiv_ddi.txt'  
matrices, labels = load_data(matrix_folder_hivddi, label_file_hivddi)
# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 4s 35ms/step - accuracy: 0.4933 - loss: 0.9262 - val_accuracy: 0.5200 - val_loss: 0.8096
Epoch 2/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5280 - loss: 0.8361 - val_accuracy: 0.5400 - val_loss: 0.7780
Epoch 3/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5343 - loss: 0.7909 - val_accuracy: 0.5000 - val_loss: 0.7646
Epoch 4/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5756 - loss: 0.7456 - val_accuracy: 0.5200 - val_loss: 0.7568
Epoch 5/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5256 - loss: 0.7886 - val_accuracy: 0.5800 - val_loss: 0.7495
Epoch 6/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5543 - loss: 0.7544 - val_accuracy: 0.6200 - val_loss: 0.7424
Epoch 7/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5805 - loss: 0.7511 - val_accuracy: 0.6000 - val_loss: 0.7370
Epoch 8/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6040 - loss: 0.7329 - v

In [21]:
save_results("hiv_ddi_50", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/classification_results2_FNN.json


In [22]:
save_classification_report("hiv_ddi_50", avg_classification_report)

Classification report saved to reports\classification_reports2_ffnn.json


In [23]:
# Load dataset
matrix_folder_hivlpv = 'data/matrices/hiv_lpv_matrix'  
label_file_hivlpv = 'data/labels/hiv_lpv.txt' 
matrices, labels = load_data(matrix_folder_hivlpv, label_file_hivlpv)

# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 3s 51ms/step - accuracy: 0.4801 - loss: 1.0132 - val_accuracy: 0.5000 - val_loss: 0.8617
Epoch 2/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.4604 - loss: 0.9196 - val_accuracy: 0.5000 - val_loss: 0.7884
Epoch 3/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.4778 - loss: 0.8163 - val_accuracy: 0.7250 - val_loss: 0.7611
Epoch 4/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.4941 - loss: 0.7906 - val_accuracy: 0.5000 - val_loss: 0.7541
Epoch 5/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5260 - loss: 0.7905 - val_accuracy: 0.5000 - val_loss: 0.7508
Epoch 6/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5392 - loss: 0.7710 - val_accuracy: 0.5000 - val_loss: 0.7461
Epoch 7/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5323 - loss: 0.7848 - val_accuracy: 0.5000 - val_loss: 0.7405
Epoch 8/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5328 - loss: 0.7701 - v

In [ ]:
save_results("hiv_rtv_50", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/classification_results2_FNN.json


In [25]:
save_classification_report("hiv_lpv_50", avg_classification_report)

Classification report saved to reports\classification_reports2_ffnn.json


In [9]:
# Load dataset
matrix_folder_hivrtv = 'data/matrices/hiv_rtv_matrix'  
label_file_hivrtv = 'data/labels/hiv_rtv.txt' 
matrices, labels = load_data(matrix_folder_hivrtv, label_file_hivrtv)

# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)
save_results("hiv_rtv_50", fold_accuracies, fold_roc_aucs, fold_reports)  
save_classification_report("hiv_rtv_50", avg_classification_report)


Training Fold 1/10...
Epoch 1/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - accuracy: 0.4967 - loss: 0.9667 - val_accuracy: 0.5763 - val_loss: 0.7748
Epoch 2/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5330 - loss: 0.8422 - val_accuracy: 0.5763 - val_loss: 0.7339
Epoch 3/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5740 - loss: 0.7469 - val_accuracy: 0.7627 - val_loss: 0.7070
Epoch 4/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6110 - loss: 0.7381 - val_accuracy: 0.7797 - val_loss: 0.6756
Epoch 5/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6136 - loss: 0.7213 - val_accuracy: 0.7627 - val_loss: 0.6431
Epoch 6/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7004 - loss: 0.6528 - val_accuracy: 0.7627 - val_loss: 0.6142
Epoch 7/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7539 - loss: 0.6024 - val_accuracy: 0.7797 - val_loss: 0.5884
Epoch 8/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7664 - loss: 0.5860 - v

RES 25

In [26]:
# Load dataset
matrix_folder_antiinflam = 'data/matrices/mat_res25/aip_antiinflam_matrix'  
label_file_antiinflam = 'data/labels/aip_antiinflam.txt' 
matrices, labels = load_data(matrix_folder_antiinflam, label_file_antiinflam)

# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.5538 - loss: 0.8723 - val_accuracy: 0.5941 - val_loss: 0.7691
Epoch 2/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.5451 - loss: 0.7859 - val_accuracy: 0.5941 - val_loss: 0.7401
Epoch 3/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5627 - loss: 0.7733 - val_accuracy: 0.5941 - val_loss: 0.7255
Epoch 4/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5361 - loss: 0.7544 - val_accuracy: 0.5941 - val_loss: 0.7158
Epoch 5/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5342 - loss: 0.7411 - val_accuracy: 0.5941 - val_loss: 0.7068
Epoch 6/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6058 - loss: 0.7124 - val_accuracy: 0.5941 - val_loss: 0.6970
Epoch 7/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5948 - loss: 0.7072 - val_accuracy: 0.5882 - val_loss: 0.6891
Epoch 8/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6116 - loss: 0.6905 - val_accu

In [27]:
save_results("aip_antiinflam_25", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/classification_results2_FNN.json


In [28]:
save_classification_report("aip_antiinflam_25", avg_classification_report)

Classification report saved to reports\classification_reports2_ffnn.json


In [29]:
# Load dataset
matrix_folder_antipb = 'data/matrices/mat_res25/amp_antibp_matrix'
label_file_antipb = 'data/labels/amp_antibp.txt'
matrices, labels = load_data(matrix_folder_antipb, label_file_antipb)
# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - accuracy: 0.5000 - loss: 1.0182 - val_accuracy: 0.5072 - val_loss: 0.8581
Epoch 2/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5230 - loss: 0.8918 - val_accuracy: 0.5072 - val_loss: 0.7942
Epoch 3/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5080 - loss: 0.8280 - val_accuracy: 0.7826 - val_loss: 0.7682
Epoch 4/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5179 - loss: 0.7980 - val_accuracy: 0.8551 - val_loss: 0.7501
Epoch 5/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5301 - loss: 0.7998 - val_accuracy: 0.7971 - val_loss: 0.7332
Epoch 6/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5984 - loss: 0.7599 - val_accuracy: 0.7826 - val_loss: 0.7155
Epoch 7/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6036 - loss: 0.7483 - val_accuracy: 0.7971 - val_loss: 0.6944
Epoch 8/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6293 - loss: 0.7168 - val_ac

In [30]:
save_results("amp_antibp_25", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/classification_results2_FNN.json


In [31]:
save_classification_report("amp_antibp_25", avg_classification_report)

Classification report saved to reports\classification_reports2_ffnn.json


In [32]:
# Load dataset
matrix_folder_antipb2 = 'data/matrices/mat_res25/amp_antibp2_matrix' 
label_file_antipb2 = 'data/labels/amp_antibp2.txt'  
matrices, labels = load_data(matrix_folder_antipb2, label_file_antipb2)
# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.4765 - loss: 0.9079 - val_accuracy: 0.7563 - val_loss: 0.7830
Epoch 2/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5253 - loss: 0.7993 - val_accuracy: 0.7812 - val_loss: 0.7497
Epoch 3/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5540 - loss: 0.7603 - val_accuracy: 0.8125 - val_loss: 0.7214
Epoch 4/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5848 - loss: 0.7330 - val_accuracy: 0.8000 - val_loss: 0.6900
Epoch 5/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6357 - loss: 0.7032 - val_accuracy: 0.8125 - val_loss: 0.6471
Epoch 6/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7232 - loss: 0.6451 - val_accuracy: 0.8188 - val_loss: 0.6128
Epoch 7/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7835 - loss: 0.5951 - val_accuracy: 0.8125 - val_loss: 0.5891
Epoch 8/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7898 - loss: 0.5710 - val_accu

In [33]:
save_results("amp_antibp2_25", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/classification_results2_FNN.json


In [34]:
save_classification_report("amp_antibp2_25", avg_classification_report)

Classification report saved to reports\classification_reports2_ffnn.json


In [35]:
# Load dataset
matrix_folder_csamp = 'data/matrices/mat_res25/amp_csamp_matrix'  
label_file_csamp = 'data/labels/amp_csamp.txt'  
matrices, labels = load_data(matrix_folder_csamp, label_file_csamp)
# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 79ms/step - accuracy: 0.5192 - loss: 0.8960 - val_accuracy: 0.4286 - val_loss: 0.8819
Epoch 2/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.4929 - loss: 0.9379 - val_accuracy: 0.4286 - val_loss: 0.8557
Epoch 3/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.5366 - loss: 0.8869 - val_accuracy: 0.4762 - val_loss: 0.8339
Epoch 4/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.4416 - loss: 0.9084 - val_accuracy: 0.6190 - val_loss: 0.8172
Epoch 5/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.4542 - loss: 0.8729 - val_accuracy: 0.6190 - val_loss: 0.8037
Epoch 6/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.5078 - loss: 0.8433 - val_accuracy: 0.5714 - val_loss: 0.7933
Epoch 7/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.5198 - loss: 0.7888 - val_accuracy: 0.6190 - val_loss: 0.7846
Epoch 8/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.5011 - loss: 0.8460 - val_accuracy: 0.6

In [36]:
save_results("amp_csamp_25", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/classification_results2_FNN.json


In [37]:
save_classification_report("amp_csamp_25", avg_classification_report)

Classification report saved to reports\classification_reports2_ffnn.json


In [38]:
# Load dataset
matrix_folder_hivddi = 'data/matrices/mat_res25/hiv_ddi_matrix' 
label_file_hivddi = 'data/labels/hiv_ddi.txt'  
matrices, labels = load_data(matrix_folder_hivddi, label_file_hivddi)
# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - accuracy: 0.5088 - loss: 0.9918 - val_accuracy: 0.4800 - val_loss: 0.9056
Epoch 2/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5328 - loss: 0.8598 - val_accuracy: 0.4800 - val_loss: 0.8358
Epoch 3/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.4773 - loss: 0.8562 - val_accuracy: 0.4800 - val_loss: 0.7993
Epoch 4/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5172 - loss: 0.8285 - val_accuracy: 0.4800 - val_loss: 0.7818
Epoch 5/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5058 - loss: 0.8220 - val_accuracy: 0.4800 - val_loss: 0.7708
Epoch 6/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5402 - loss: 0.7933 - val_accuracy: 0.5000 - val_loss: 0.7621
Epoch 7/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5548 - loss: 0.7699 - val_accuracy: 0.5400 - val_loss: 0.7556
Epoch 8/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5589 - loss: 0.7668 - v

In [39]:
save_results("hiv_ddi_25", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/classification_results2_FNN.json


In [40]:
save_classification_report("hiv_ddi_25", avg_classification_report)

Classification report saved to reports\classification_reports2_ffnn.json


In [41]:
# Load dataset
matrix_folder_hivlpv = 'data/matrices/mat_res25/hiv_lpv_matrix'  
label_file_hivlpv = 'data/labels/hiv_lpv.txt' 
matrices, labels = load_data(matrix_folder_hivlpv, label_file_hivlpv)

# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 3s 54ms/step - accuracy: 0.4863 - loss: 0.9433 - val_accuracy: 0.5000 - val_loss: 0.8407
Epoch 2/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.4769 - loss: 0.8927 - val_accuracy: 0.5000 - val_loss: 0.8102
Epoch 3/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5007 - loss: 0.8603 - val_accuracy: 0.5000 - val_loss: 0.7934
Epoch 4/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5084 - loss: 0.8426 - val_accuracy: 0.5000 - val_loss: 0.7785
Epoch 5/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5384 - loss: 0.7863 - val_accuracy: 0.5000 - val_loss: 0.7679
Epoch 6/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.5764 - loss: 0.7759 - val_accuracy: 0.5000 - val_loss: 0.7588
Epoch 7/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.5427 - loss: 0.7697 - val_accuracy: 0.5000 - val_loss: 0.7493
Epoch 8/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.5394 - loss: 0.7835 - v

In [42]:
save_results("hiv_lpv_25", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/classification_results2_FNN.json


In [43]:
save_classification_report("hiv_lpv_25", avg_classification_report)

Classification report saved to reports\classification_reports2_ffnn.json


In [10]:
# Load dataset
matrix_folder_hivrtv = 'data/matrices/mat_res25/hiv_rtv_matrix'  
label_file_hivrtv = 'data/labels/hiv_rtv.txt' 
matrices, labels = load_data(matrix_folder_hivrtv, label_file_hivrtv)

# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)
save_results("hiv_rtv_25", fold_accuracies, fold_roc_aucs, fold_reports)  
save_classification_report("hiv_rtv_25", avg_classification_report)


Training Fold 1/10...
Epoch 1/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - accuracy: 0.4961 - loss: 0.9637 - val_accuracy: 0.4237 - val_loss: 0.8405
Epoch 2/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5307 - loss: 0.8636 - val_accuracy: 0.6102 - val_loss: 0.7845
Epoch 3/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5762 - loss: 0.8008 - val_accuracy: 0.5763 - val_loss: 0.7559
Epoch 4/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5273 - loss: 0.8099 - val_accuracy: 0.7797 - val_loss: 0.7378
Epoch 5/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5370 - loss: 0.8018 - val_accuracy: 0.7797 - val_loss: 0.7208
Epoch 6/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5681 - loss: 0.7572 - val_accuracy: 0.7797 - val_loss: 0.6969
Epoch 7/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5989 - loss: 0.7272 - val_accuracy: 0.7797 - val_loss: 0.6692
Epoch 8/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6294 - loss: 0.6888 - val_accu

RES 75

In [44]:
# Load dataset
matrix_folder_antiinflam = 'data/matrices/mat_res75/aip_antiinflam_matrix'
label_file_antiinflam = 'data/labels/aip_antiinflam.txt' 
matrices, labels = load_data(matrix_folder_antiinflam, label_file_antiinflam)
# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 14s 23ms/step - accuracy: 0.5240 - loss: 0.8471 - val_accuracy: 0.5941 - val_loss: 0.7453
Epoch 2/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.5513 - loss: 0.7734 - val_accuracy: 0.5941 - val_loss: 0.7331
Epoch 3/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.6045 - loss: 0.7293 - val_accuracy: 0.5941 - val_loss: 0.7229
Epoch 4/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.6139 - loss: 0.7146 - val_accuracy: 0.6176 - val_loss: 0.7146
Epoch 5/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6612 - loss: 0.6717 - val_accuracy: 0.6294 - val_loss: 0.7077
Epoch 6/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.7427 - loss: 0.6153 - val_accuracy: 0.6529 - val_loss: 0.7167
Epoch 7/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.7940 - loss: 0.5643 - val_accuracy: 0.6647 - val_loss: 0.7410
Epoch 8/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.8340 - loss: 0.5253 - 

In [45]:
save_results("aip_antiinflam_75", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/classification_results2_FNN.json


In [46]:
save_classification_report("aip_antiinflam_75", avg_classification_report)

Classification report saved to reports\classification_reports2_ffnn.json


In [47]:
# Load dataset
matrix_folder_antipb = 'data/matrices/mat_res75/amp_antibp_matrix'
label_file_antipb = 'data/labels/amp_antibp.txt'
matrices, labels = load_data(matrix_folder_antipb, label_file_antipb)
# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - accuracy: 0.5175 - loss: 0.8780 - val_accuracy: 0.4928 - val_loss: 0.7744
Epoch 2/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.4980 - loss: 0.8025 - val_accuracy: 0.7681 - val_loss: 0.7459
Epoch 3/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5438 - loss: 0.7529 - val_accuracy: 0.7826 - val_loss: 0.7262
Epoch 4/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.6113 - loss: 0.7204 - val_accuracy: 0.7971 - val_loss: 0.7024
Epoch 5/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7151 - loss: 0.6738 - val_accuracy: 0.8116 - val_loss: 0.6721
Epoch 6/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7661 - loss: 0.6493 - val_accuracy: 0.7971 - val_loss: 0.6358
Epoch 7/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8268 - loss: 0.5979 - val_accuracy: 0.8116 - val_loss: 0.5987
Epoch 8/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8602 - loss: 0.5436 - v

In [48]:
save_results("amp_antibp_75", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/classification_results2_FNN.json


In [49]:
save_classification_report("amp_antibp_75", avg_classification_report)

Classification report saved to reports\classification_reports2_ffnn.json


In [50]:
# Load dataset
matrix_folder_antipb = 'data/matrices/mat_res75/amp_antibp2_matrix'
label_file_antipb = 'data/labels/amp_antibp2.txt'
matrices, labels = load_data(matrix_folder_antipb, label_file_antipb)
# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - accuracy: 0.4896 - loss: 0.8994 - val_accuracy: 0.7875 - val_loss: 0.7492
Epoch 2/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.5731 - loss: 0.7765 - val_accuracy: 0.7875 - val_loss: 0.7136
Epoch 3/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.5924 - loss: 0.7397 - val_accuracy: 0.8250 - val_loss: 0.6460
Epoch 4/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.7288 - loss: 0.6202 - val_accuracy: 0.8000 - val_loss: 0.5960
Epoch 5/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.8382 - loss: 0.5389 - val_accuracy: 0.7937 - val_loss: 0.5886
Epoch 6/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.8890 - loss: 0.4628 - val_accuracy: 0.7625 - val_loss: 0.6155
Epoch 7/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.8992 - loss: 0.4147 - val_accuracy: 0.7500 - val_loss: 0.6694
Epoch 8/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9167 - loss: 0.3857 - v

In [51]:
save_results("amp_antibp2_75", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/classification_results2_FNN.json


In [52]:
save_classification_report("amp_antibp2_75", avg_classification_report)

Classification report saved to reports\classification_reports2_ffnn.json


In [53]:
# Load dataset
matrix_folder_csamp = 'data/matrices/mat_res75/amp_csamp_matrix'  
label_file_csamp = 'data/labels/amp_csamp.txt'  
matrices, labels = load_data(matrix_folder_csamp, label_file_csamp)
# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 110ms/step - accuracy: 0.4365 - loss: 0.9453 - val_accuracy: 0.5238 - val_loss: 0.8355
Epoch 2/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.4898 - loss: 0.8676 - val_accuracy: 0.4286 - val_loss: 0.8020
Epoch 3/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.5363 - loss: 0.7984 - val_accuracy: 0.6190 - val_loss: 0.7811
Epoch 4/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.5874 - loss: 0.7721 - val_accuracy: 0.5714 - val_loss: 0.7688
Epoch 5/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.5011 - loss: 0.7942 - val_accuracy: 0.5714 - val_loss: 0.7620
Epoch 6/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.5932 - loss: 0.7566 - val_accuracy: 0.5714 - val_loss: 0.7578
Epoch 7/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.5832 - loss: 0.7475 - val_accuracy: 0.6667 - val_loss: 0.7551
Epoch 8/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.6705 - loss: 0.7121 - val_accuracy: 0.

In [54]:
save_results("amp_csamp_75", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/classification_results2_FNN.json


In [55]:
save_classification_report("amp_csamp_75", avg_classification_report)

Classification report saved to reports\classification_reports2_ffnn.json


In [56]:
# Load dataset
matrix_folder_hivddi = 'data/matrices/mat_res75/hiv_ddi_matrix' 
label_file_hivddi = 'data/labels/hiv_ddi.txt'  
matrices, labels = load_data(matrix_folder_hivddi, label_file_hivddi)
# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - accuracy: 0.5262 - loss: 1.0980 - val_accuracy: 0.4800 - val_loss: 0.9867
Epoch 2/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.5218 - loss: 0.9027 - val_accuracy: 0.4800 - val_loss: 0.8682
Epoch 3/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.5330 - loss: 0.8279 - val_accuracy: 0.4800 - val_loss: 0.8098
Epoch 4/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5349 - loss: 0.7906 - val_accuracy: 0.4800 - val_loss: 0.7812
Epoch 5/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5301 - loss: 0.7553 - val_accuracy: 0.4800 - val_loss: 0.7662
Epoch 6/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.5760 - loss: 0.7511 - val_accuracy: 0.4800 - val_loss: 0.7564
Epoch 7/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.5754 - loss: 0.7262 - val_accuracy: 0.4800 - val_loss: 0.7500
Epoch 8/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.6253 - loss: 0.7235 - v

In [57]:
save_results("hiv_ddi_75", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/classification_results2_FNN.json


In [58]:
save_classification_report("hiv_ddi_75", avg_classification_report)

Classification report saved to reports\classification_reports2_ffnn.json


In [59]:
# Load dataset
matrix_folder_hivlpv = 'data/matrices/mat_res75/hiv_lpv_matrix'  
label_file_hivlpv = 'data/labels/hiv_lpv.txt' 
matrices, labels = load_data(matrix_folder_hivlpv, label_file_hivlpv)

# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 3s 45ms/step - accuracy: 0.4753 - loss: 0.9951 - val_accuracy: 0.5000 - val_loss: 0.8526
Epoch 2/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.4953 - loss: 0.8769 - val_accuracy: 0.5000 - val_loss: 0.7815
Epoch 3/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.4695 - loss: 0.8088 - val_accuracy: 0.5000 - val_loss: 0.7579
Epoch 4/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5379 - loss: 0.7633 - val_accuracy: 0.7250 - val_loss: 0.7493
Epoch 5/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5319 - loss: 0.7593 - val_accuracy: 0.6000 - val_loss: 0.7451
Epoch 6/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5836 - loss: 0.7475 - val_accuracy: 0.5500 - val_loss: 0.7414
Epoch 7/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6017 - loss: 0.7306 - val_accuracy: 0.5250 - val_loss: 0.7370
Epoch 8/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5265 - loss: 0.7566 - v

In [60]:
save_results("hiv_lpv_75", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/classification_results2_FNN.json


In [61]:
save_classification_report("hiv_lpv_75", avg_classification_report)

Classification report saved to reports\classification_reports2_ffnn.json


In [11]:
# Load dataset
matrix_folder_hivrtv = 'data/matrices/mat_res75/hiv_rtv_matrix'  
label_file_hivrtv = 'data/labels/hiv_rtv.txt' 
matrices, labels = load_data(matrix_folder_hivrtv, label_file_hivrtv)

# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)
save_results("hiv_rtv_75", fold_accuracies, fold_roc_aucs, fold_reports)  
save_classification_report("hiv_rtv_75", avg_classification_report)


Training Fold 1/10...
Epoch 1/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 41ms/step - accuracy: 0.5189 - loss: 0.8989 - val_accuracy: 0.7627 - val_loss: 0.7657
Epoch 2/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5873 - loss: 0.7715 - val_accuracy: 0.7458 - val_loss: 0.7277
Epoch 3/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5536 - loss: 0.7714 - val_accuracy: 0.7458 - val_loss: 0.7056
Epoch 4/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6192 - loss: 0.7154 - val_accuracy: 0.7458 - val_loss: 0.6808
Epoch 5/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6604 - loss: 0.7026 - val_accuracy: 0.7627 - val_loss: 0.6548
Epoch 6/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7376 - loss: 0.6436 - val_accuracy: 0.7458 - val_loss: 0.6281
Epoch 7/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7538 - loss: 0.6223 - val_accuracy: 0.7627 - val_loss: 0.6070
Epoch 8/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8045 - loss: 0.5710 - v

RES 100

In [7]:
# Load dataset
matrix_folder_antiinflam = 'data/matrices/mat_res100/aip_antiinflam_matrix'  
label_file_antiinflam = 'data/labels/aip_antiinflam.txt' 
matrices, labels = load_data(matrix_folder_antiinflam, label_file_antiinflam)
# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 8s 29ms/step - accuracy: 0.4443 - loss: 0.9745 - val_accuracy: 0.5941 - val_loss: 0.7587
Epoch 2/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - accuracy: 0.5554 - loss: 0.7806 - val_accuracy: 0.5941 - val_loss: 0.7418
Epoch 3/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.5730 - loss: 0.7502 - val_accuracy: 0.6000 - val_loss: 0.7312
Epoch 4/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.6154 - loss: 0.7189 - val_accuracy: 0.6000 - val_loss: 0.7243
Epoch 5/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.6736 - loss: 0.6722 - val_accuracy: 0.6118 - val_loss: 0.7247
Epoch 6/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.7806 - loss: 0.6142 - val_accuracy: 0.6353 - val_loss: 0.7346
Epoch 7/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - accuracy: 0.8426 - loss: 0.5433 - val_accuracy: 0.6529 - val_loss: 0.7628
Epoch 8/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.9001 - loss: 0.4786 - v

In [8]:
save_results("aip_antiinflam_100", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/classification_results2_FNN.json


In [9]:
save_classification_report("aip_antiinflam_100", avg_classification_report)

Classification report saved to reports\classification_reports2_ffnn.json


In [10]:
# Load dataset
matrix_folder_antipb = 'data/matrices/mat_res100/amp_antibp_matrix'
label_file_antipb = 'data/labels/amp_antibp.txt'
matrices, labels = load_data(matrix_folder_antipb, label_file_antipb)
# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 53ms/step - accuracy: 0.5175 - loss: 0.8744 - val_accuracy: 0.6087 - val_loss: 0.7598
Epoch 2/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.5413 - loss: 0.7717 - val_accuracy: 0.8551 - val_loss: 0.7369
Epoch 3/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.6011 - loss: 0.7351 - val_accuracy: 0.8261 - val_loss: 0.7076
Epoch 4/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.6401 - loss: 0.7034 - val_accuracy: 0.8406 - val_loss: 0.6700
Epoch 5/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.7583 - loss: 0.6362 - val_accuracy: 0.8551 - val_loss: 0.6272
Epoch 6/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.8722 - loss: 0.5508 - val_accuracy: 0.8406 - val_loss: 0.5859
Epoch 7/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.8755 - loss: 0.5073 - val_accuracy: 0.8261 - val_loss: 0.5525
Epoch 8/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.9235 - loss: 0.4403 - v

In [11]:
save_results("amp_antibp_100", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/classification_results2_FNN.json


In [12]:
save_classification_report("amp_antibp_100", avg_classification_report)

Classification report saved to reports\classification_reports2_ffnn.json


In [13]:
# Load dataset
matrix_folder_antipb = 'data/matrices/mat_res100/amp_antibp2_matrix'
label_file_antipb = 'data/labels/amp_antibp2.txt'
matrices, labels = load_data(matrix_folder_antipb, label_file_antipb)
# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 5s 32ms/step - accuracy: 0.4883 - loss: 0.9213 - val_accuracy: 0.6438 - val_loss: 0.7570
Epoch 2/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.5688 - loss: 0.7543 - val_accuracy: 0.8125 - val_loss: 0.7054
Epoch 3/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.6931 - loss: 0.6709 - val_accuracy: 0.8062 - val_loss: 0.6472
Epoch 4/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - accuracy: 0.7767 - loss: 0.6077 - val_accuracy: 0.7875 - val_loss: 0.6037
Epoch 5/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.8468 - loss: 0.5303 - val_accuracy: 0.7937 - val_loss: 0.5873
Epoch 6/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.9109 - loss: 0.4359 - val_accuracy: 0.7812 - val_loss: 0.5979
Epoch 7/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.9405 - loss: 0.3687 - val_accuracy: 0.7688 - val_loss: 0.6529
Epoch 8/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - accuracy: 0.9739 - loss: 0.3206 - v

In [14]:
save_results("amp_antibp2_100", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/classification_results2_FNN.json


In [15]:
save_classification_report("amp_antibp2_100", avg_classification_report)

Classification report saved to reports\classification_reports2_ffnn.json


In [16]:
# Load dataset
matrix_folder_csamp = 'data/matrices/mat_res100/amp_csamp_matrix'  
label_file_csamp = 'data/labels/amp_csamp.txt'  
matrices, labels = load_data(matrix_folder_csamp, label_file_csamp)
# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 103ms/step - accuracy: 0.4625 - loss: 1.0139 - val_accuracy: 0.4286 - val_loss: 0.9647
Epoch 2/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.5186 - loss: 0.8776 - val_accuracy: 0.4286 - val_loss: 0.8826
Epoch 3/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.5871 - loss: 0.8216 - val_accuracy: 0.4286 - val_loss: 0.8330
Epoch 4/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.5618 - loss: 0.8111 - val_accuracy: 0.4286 - val_loss: 0.8040
Epoch 5/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.6273 - loss: 0.7056 - val_accuracy: 0.5714 - val_loss: 0.7864
Epoch 6/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.5465 - loss: 0.7563 - val_accuracy: 0.5714 - val_loss: 0.7740
Epoch 7/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.6170 - loss: 0.7383 - val_accuracy: 0.5714 - val_loss: 0.7635
Epoch 8/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.6348 - loss: 0.7158 - val_accuracy: 0.

In [17]:
save_results("amp_csamp_100", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/classification_results2_FNN.json


In [18]:
save_classification_report("amp_csamp_100", avg_classification_report)

Classification report saved to reports\classification_reports2_ffnn.json


In [19]:
# Load dataset
matrix_folder_hivddi = 'data/matrices/mat_res100/hiv_ddi_matrix' 
label_file_hivddi = 'data/labels/hiv_ddi.txt'  
matrices, labels = load_data(matrix_folder_hivddi, label_file_hivddi)
# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 4s 56ms/step - accuracy: 0.5289 - loss: 1.0784 - val_accuracy: 0.4800 - val_loss: 0.9674
Epoch 2/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.5303 - loss: 0.9056 - val_accuracy: 0.4800 - val_loss: 0.8625
Epoch 3/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.5415 - loss: 0.8170 - val_accuracy: 0.4800 - val_loss: 0.8103
Epoch 4/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.5172 - loss: 0.7905 - val_accuracy: 0.4800 - val_loss: 0.7813
Epoch 5/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.5522 - loss: 0.7514 - val_accuracy: 0.4800 - val_loss: 0.7667
Epoch 6/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.5965 - loss: 0.7238 - val_accuracy: 0.4800 - val_loss: 0.7562
Epoch 7/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.5658 - loss: 0.7440 - val_accuracy: 0.5200 - val_loss: 0.7482
Epoch 8/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.6068 - loss: 0.7197 - v

In [20]:
save_results("hiv_ddi_100", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/classification_results2_FNN.json


In [21]:
save_classification_report("hiv_ddi_100", avg_classification_report)

Classification report saved to reports\classification_reports2_ffnn.json


In [22]:
# Load dataset
matrix_folder_hivlpv = 'data/matrices/mat_res100/hiv_lpv_matrix'  
label_file_hivlpv = 'data/labels/hiv_lpv.txt' 
matrices, labels = load_data(matrix_folder_hivlpv, label_file_hivlpv)

# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 5s 116ms/step - accuracy: 0.5120 - loss: 0.9620 - val_accuracy: 0.5000 - val_loss: 0.7875
Epoch 2/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - accuracy: 0.4563 - loss: 0.8428 - val_accuracy: 0.7250 - val_loss: 0.7491
Epoch 3/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.5151 - loss: 0.8012 - val_accuracy: 0.5500 - val_loss: 0.7429
Epoch 4/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.5102 - loss: 0.7953 - val_accuracy: 0.5000 - val_loss: 0.7383
Epoch 5/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.5596 - loss: 0.7419 - val_accuracy: 0.5250 - val_loss: 0.7328
Epoch 6/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.5570 - loss: 0.7666 - val_accuracy: 0.5250 - val_loss: 0.7263
Epoch 7/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.5789 - loss: 0.7350 - val_accuracy: 0.5750 - val_loss: 0.7203
Epoch 8/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.6331 - loss: 0.7054 - 

In [23]:
save_results("hiv_lpv_100", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/classification_results2_FNN.json


In [24]:
save_classification_report("hiv_lpv_100", avg_classification_report)

Classification report saved to reports\classification_reports2_ffnn.json


In [12]:
# Load dataset
matrix_folder_hivrtv = 'data/matrices/mat_res100/hiv_rtv_matrix'  
label_file_hivrtv = 'data/labels/hiv_rtv.txt' 
matrices, labels = load_data(matrix_folder_hivrtv, label_file_hivrtv)

# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)
save_results("hiv_rtv_100", fold_accuracies, fold_roc_aucs, fold_reports)  
save_classification_report("hiv_rtv_100", avg_classification_report)


Training Fold 1/10...
Epoch 1/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 6s 151ms/step - accuracy: 0.5241 - loss: 0.8837 - val_accuracy: 0.5254 - val_loss: 0.7536
Epoch 2/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.5524 - loss: 0.7682 - val_accuracy: 0.7627 - val_loss: 0.7104
Epoch 3/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.6013 - loss: 0.7195 - val_accuracy: 0.7797 - val_loss: 0.6805
Epoch 4/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.6679 - loss: 0.6775 - val_accuracy: 0.7797 - val_loss: 0.6527
Epoch 5/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - accuracy: 0.7674 - loss: 0.6246 - val_accuracy: 0.7797 - val_loss: 0.6263
Epoch 6/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.7545 - loss: 0.5988 - val_accuracy: 0.7797 - val_loss: 0.6057
Epoch 7/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.7845 - loss: 0.5673 - val_accuracy: 0.7797 - val_loss: 0.5918
Epoch 8/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.8336 - loss: 0.5547 - 